# 🏦 Bank Fraud Detection — Exploratory Data Analysis

**Dataset:** 1,000,000 bank transactions across 10 countries  
**Goal:** Understand fraud patterns, high-risk segments, and behavioral signals

| Feature | Detail |
|---------|--------|
| Rows | 1,000,000 transactions |
| Columns | 26 features |
| Fraud Rate | ~5.5% (55,255 fraudulent transactions) |
| Fraud Types | 6 types (Phishing, Account Takeover, Card Cloning, etc.) |

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import os
os.makedirs('outputs', exist_ok=True)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully!')


## 1. Load & Preview Data

In [ ]:
df = pd.read_csv(r"D:\bank_fraud\bank_fraud.csv")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


In [ ]:
print("Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print("Note: fraud_type is NaN for non-fraud rows — expected!")
df['fraud_type'] = df['fraud_type'].fillna('Not Fraud')


## 2. Statistical Summary

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues').format(precision=2)


## 3. Fraud Overview

In [ ]:
fraud_rate = df['is_fraud'].mean() * 100
fraud_count = df['is_fraud'].sum()
legit_count = len(df) - fraud_count

print(f"Total Transactions : {len(df):,}")
print(f"Fraudulent         : {fraud_count:,} ({fraud_rate:.2f}%)")
print(f"Legitimate         : {legit_count:,} ({100-fraud_rate:.2f}%)")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Pie chart
axes[0].pie([legit_count, fraud_count],
            labels=['Legitimate', 'Fraudulent'],
            autopct='%1.2f%%',
            colors=['#2ecc71', '#e74c3c'],
            startangle=90,
            explode=(0, 0.08),
            textprops={'fontsize': 13})
axes[0].set_title('Fraud vs Legitimate Transactions', fontweight='bold')

# Fraud types
fraud_types = df[df['is_fraud'] == 1]['fraud_type'].value_counts()
colors_ft = sns.color_palette('Reds_r', len(fraud_types))
bars = axes[1].barh(fraud_types.index[::-1], fraud_types.values[::-1], color=colors_ft[::-1])
axes[1].set_title('Fraud Type Distribution', fontweight='bold')
axes[1].set_xlabel('Count')
for bar, val in zip(bars[::-1], fraud_types.values):
    axes[1].text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=10)

# Fraud rate by country
country_fraud = df.groupby('country')['is_fraud'].mean() * 100
country_fraud = country_fraud.sort_values(ascending=False)
colors_c = ['#e74c3c' if v > fraud_rate else '#3498db' for v in country_fraud.values]
bars2 = axes[2].bar(country_fraud.index, country_fraud.values, color=colors_c, edgecolor='white')
axes[2].axhline(fraud_rate, color='red', linestyle='--', lw=1.5,
                label=f'Overall: {fraud_rate:.2f}%')
axes[2].set_title('Fraud Rate by Country (%)', fontweight='bold')
axes[2].set_xlabel('Country')
axes[2].set_ylabel('Fraud Rate (%)')
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend()

plt.suptitle('Fraud Overview', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/fraud_overview.png', bbox_inches='tight', dpi=150)
plt.show()


## 4. Transaction Amount Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

fraud_amt   = df[df['is_fraud'] == 1]['transaction_amount']
legit_amt   = df[df['is_fraud'] == 0]['transaction_amount']

# Overlapping distribution
axes[0].hist(legit_amt, bins=60, alpha=0.6, color='#2ecc71', label='Legitimate', density=True)
axes[0].hist(fraud_amt, bins=60, alpha=0.6, color='#e74c3c', label='Fraudulent', density=True)
axes[0].set_title('Transaction Amount Distribution', fontweight='bold')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Density')
axes[0].legend()

# Boxplot
bp = axes[1].boxplot([legit_amt, fraud_amt],
                      labels=['Legitimate', 'Fraudulent'],
                      patch_artist=True,
                      notch=True)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][1].set_facecolor('#e74c3c')
for box in bp['boxes']:
    box.set_alpha(0.7)
axes[1].set_title('Transaction Amount: Fraud vs Legit', fontweight='bold')
axes[1].set_ylabel('Amount')

# Avg transaction amount by merchant category
merchant_avg = df.groupby(['merchant_category', 'is_fraud'])['transaction_amount'].mean().unstack()
merchant_avg.columns = ['Legitimate', 'Fraudulent']
merchant_avg = merchant_avg.sort_values('Fraudulent', ascending=False)
x = range(len(merchant_avg))
w = 0.35
axes[2].bar([i - w/2 for i in x], merchant_avg['Legitimate'], w,
            label='Legitimate', color='#2ecc71', alpha=0.85)
axes[2].bar([i + w/2 for i in x], merchant_avg['Fraudulent'], w,
            label='Fraudulent', color='#e74c3c', alpha=0.85)
axes[2].set_xticks(list(x))
axes[2].set_xticklabels(merchant_avg.index, rotation=45, ha='right', fontsize=9)
axes[2].set_title('Avg Amount by Merchant Category', fontweight='bold')
axes[2].set_ylabel('Avg Amount')
axes[2].legend()

plt.suptitle('Transaction Amount Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/transaction_amount.png', bbox_inches='tight', dpi=150)
plt.show()


## 5. Time-Based Fraud Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Fraud rate by hour
hour_fraud = df.groupby('hour_of_day')['is_fraud'].mean() * 100
axes[0].plot(hour_fraud.index, hour_fraud.values, color='#e74c3c',
             linewidth=2.5, marker='o', markersize=5)
axes[0].fill_between(hour_fraud.index, hour_fraud.values, alpha=0.2, color='#e74c3c')
axes[0].set_title('Fraud Rate by Hour of Day (%)', fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].axhline(df['is_fraud'].mean() * 100, color='gray', linestyle='--',
                label='Overall avg')
axes[0].legend()

# Fraud count: weekday vs weekend
weekend_fraud = df.groupby('is_weekend')['is_fraud'].agg(['sum', 'mean'])
labels_w = ['Weekday', 'Weekend']
bars = axes[1].bar(labels_w, weekend_fraud['mean'] * 100,
                   color=['#3498db', '#e74c3c'], edgecolor='white')
axes[1].set_title('Fraud Rate: Weekday vs Weekend (%)', fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')
for bar, val in zip(bars, weekend_fraud['mean'].values * 100):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=12)

# Night transaction fraud
night_fraud = df.groupby('is_night_transaction')['is_fraud'].mean() * 100
labels_n = ['Day Transaction', 'Night Transaction']
colors_n = ['#f39c12', '#8e44ad']
bars2 = axes[2].bar(labels_n, night_fraud.values, color=colors_n, edgecolor='white')
axes[2].set_title('Fraud Rate: Day vs Night (%)', fontweight='bold')
axes[2].set_ylabel('Fraud Rate (%)')
for bar, val in zip(bars2, night_fraud.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=12)

plt.suptitle('Time-Based Fraud Patterns', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/time_patterns.png', bbox_inches='tight', dpi=150)
plt.show()


## 6. Payment Method & Device Type Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Fraud rate by payment method
pm_fraud = df.groupby('payment_method')['is_fraud'].mean() * 100
pm_fraud = pm_fraud.sort_values(ascending=False)
overall = df['is_fraud'].mean() * 100
colors_pm = ['#e74c3c' if v > overall else '#2ecc71' for v in pm_fraud.values]
bars = axes[0].bar(pm_fraud.index, pm_fraud.values, color=colors_pm, edgecolor='white')
axes[0].axhline(overall, color='gray', linestyle='--', lw=1.5, label=f'Overall: {overall:.2f}%')
axes[0].set_title('Fraud Rate by Payment Method (%)', fontweight='bold')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend()
for bar, val in zip(bars, pm_fraud.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

# Fraud rate by device type
dt_fraud = df.groupby('device_type')['is_fraud'].mean() * 100
dt_fraud = dt_fraud.sort_values(ascending=False)
colors_dt = ['#e74c3c' if v > overall else '#2ecc71' for v in dt_fraud.values]
bars2 = axes[1].bar(dt_fraud.index, dt_fraud.values, color=colors_dt, edgecolor='white')
axes[1].axhline(overall, color='gray', linestyle='--', lw=1.5, label=f'Overall: {overall:.2f}%')
axes[1].set_title('Fraud Rate by Device Type (%)', fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].legend()
for bar, val in zip(bars2, dt_fraud.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

plt.suptitle('Payment Method & Device Type Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/payment_device.png', bbox_inches='tight', dpi=150)
plt.show()


## 7. Merchant Category Fraud Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 8))

# Fraud rate by merchant category
mc_fraud = df.groupby('merchant_category')['is_fraud'].mean() * 100
mc_fraud = mc_fraud.sort_values(ascending=False)
colors_mc = ['#e74c3c' if v > overall else '#2ecc71' for v in mc_fraud.values]
axes[0].barh(mc_fraud.index[::-1], mc_fraud.values[::-1], color=colors_mc[::-1])
axes[0].axvline(overall, color='gray', linestyle='--', lw=1.5, label=f'Overall: {overall:.2f}%')
axes[0].set_title('Fraud Rate by Merchant Category (%)', fontweight='bold')
axes[0].set_xlabel('Fraud Rate (%)')
axes[0].legend()

# Fraud volume by merchant category
mc_volume = df[df['is_fraud'] == 1]['merchant_category'].value_counts()
colors_mv = sns.color_palette('Reds_r', len(mc_volume))
axes[1].barh(mc_volume.index[::-1], mc_volume.values[::-1], color=colors_mv[::-1])
axes[1].set_title('Fraud Volume by Merchant Category', fontweight='bold')
axes[1].set_xlabel('Fraud Count')
for i, val in enumerate(mc_volume.values[::-1]):
    axes[1].text(val + 10, i, f'{val:,}', va='center', fontsize=9)

plt.suptitle('Merchant Category Fraud Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/merchant_analysis.png', bbox_inches='tight', dpi=150)
plt.show()


## 8. Customer Profile Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()

# Age distribution
axes[0].hist(df[df['is_fraud']==0]['customer_age'], bins=40, alpha=0.6,
             color='#2ecc71', label='Legitimate', density=True)
axes[0].hist(df[df['is_fraud']==1]['customer_age'], bins=40, alpha=0.6,
             color='#e74c3c', label='Fraudulent', density=True)
axes[0].set_title('Customer Age: Fraud vs Legit', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].legend()

# Credit score
axes[1].hist(df[df['is_fraud']==0]['credit_score'], bins=40, alpha=0.6,
             color='#2ecc71', label='Legitimate', density=True)
axes[1].hist(df[df['is_fraud']==1]['credit_score'], bins=40, alpha=0.6,
             color='#e74c3c', label='Fraudulent', density=True)
axes[1].set_title('Credit Score: Fraud vs Legit', fontweight='bold')
axes[1].set_xlabel('Credit Score')
axes[1].legend()

# Account balance
axes[2].hist(df[df['is_fraud']==0]['account_balance'], bins=40, alpha=0.6,
             color='#2ecc71', label='Legitimate', density=True)
axes[2].hist(df[df['is_fraud']==1]['account_balance'], bins=40, alpha=0.6,
             color='#e74c3c', label='Fraudulent', density=True)
axes[2].set_title('Account Balance: Fraud vs Legit', fontweight='bold')
axes[2].set_xlabel('Account Balance')
axes[2].legend()

# Distance from home
axes[3].hist(df[df['is_fraud']==0]['distance_from_home_km'], bins=40, alpha=0.6,
             color='#2ecc71', label='Legitimate', density=True)
axes[3].hist(df[df['is_fraud']==1]['distance_from_home_km'], bins=40, alpha=0.6,
             color='#e74c3c', label='Fraudulent', density=True)
axes[3].set_title('Distance from Home: Fraud vs Legit', fontweight='bold')
axes[3].set_xlabel('Distance (km)')
axes[3].legend()

# Failed attempts
failed_fraud = df.groupby('failed_attempts')['is_fraud'].mean() * 100
axes[4].bar(failed_fraud.index, failed_fraud.values,
            color=sns.color_palette('Reds', len(failed_fraud)), edgecolor='white')
axes[4].set_title('Fraud Rate by Failed Attempts (%)', fontweight='bold')
axes[4].set_xlabel('Number of Failed Attempts')
axes[4].set_ylabel('Fraud Rate (%)')

# International transactions
intl_fraud = df.groupby('is_international')['is_fraud'].mean() * 100
axes[5].bar(['Domestic', 'International'], intl_fraud.values,
            color=['#3498db', '#e74c3c'], edgecolor='white')
axes[5].set_title('Fraud Rate: Domestic vs International (%)', fontweight='bold')
axes[5].set_ylabel('Fraud Rate (%)')
for i, val in enumerate(intl_fraud.values):
    axes[5].text(i, val + 0.05, f'{val:.2f}%', ha='center', va='bottom', fontsize=12)

plt.suptitle('Customer Profile Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/customer_profile.png', bbox_inches='tight', dpi=150)
plt.show()


## 9. Correlation Heatmap

In [ ]:
num_cols = ['transaction_amount', 'customer_age', 'credit_score',
            'account_age_years', 'account_balance', 'distance_from_home_km',
            'time_since_last_txn_hrs', 'failed_attempts', 'is_fraud']

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5,
            annot_kws={'size': 11}, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('outputs/correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()


## 10. Behavioral Risk Signals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# PIN changed recently
pin_fraud = df.groupby('pin_changed_recently')['is_fraud'].mean() * 100
axes[0].bar(['No', 'Yes'], pin_fraud.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Fraud Rate: PIN Changed Recently (%)', fontweight='bold')
axes[0].set_ylabel('Fraud Rate (%)')
for i, val in enumerate(pin_fraud.values):
    axes[0].text(i, val + 0.05, f'{val:.2f}%', ha='center', va='bottom', fontsize=13)

# Transaction frequency vs fraud
df['freq_bucket'] = pd.cut(df['transaction_freq_monthly'],
    bins=[0, 10, 20, 30, 50, 100],
    labels=['1-10', '11-20', '21-30', '31-50', '51+'])
freq_fraud = df.groupby('freq_bucket', observed=True)['is_fraud'].mean() * 100
axes[1].bar(freq_fraud.index, freq_fraud.values,
            color=sns.color_palette('YlOrRd', len(freq_fraud)), edgecolor='white')
axes[1].set_title('Fraud Rate by Monthly Transaction Frequency', fontweight='bold')
axes[1].set_xlabel('Transactions/Month')
axes[1].set_ylabel('Fraud Rate (%)')

# Time since last transaction
df['time_bucket'] = pd.cut(df['time_since_last_txn_hrs'],
    bins=[0, 1, 6, 24, 72, df['time_since_last_txn_hrs'].max()+1],
    labels=['<1hr', '1-6hrs', '6-24hrs', '1-3days', '>3days'])
time_fraud = df.groupby('time_bucket', observed=True)['is_fraud'].mean() * 100
axes[2].bar(time_fraud.index, time_fraud.values,
            color=sns.color_palette('Purples', len(time_fraud)), edgecolor='white')
axes[2].set_title('Fraud Rate by Time Since Last Transaction', fontweight='bold')
axes[2].set_xlabel('Time Since Last Transaction')
axes[2].set_ylabel('Fraud Rate (%)')

plt.suptitle('Behavioral Risk Signals', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/behavioral_signals.png', bbox_inches='tight', dpi=150)
plt.show()


## 11. Key Insights Summary

In [ ]:
fraud_rate      = df['is_fraud'].mean() * 100
fraud_count     = df['is_fraud'].sum()
top_fraud_type  = df[df['is_fraud']==1]['fraud_type'].value_counts().idxmax()
top_fraud_mc    = df[df['is_fraud']==1]['merchant_category'].value_counts().idxmax()
top_fraud_pm    = df.groupby('payment_method')['is_fraud'].mean().idxmax()
top_fraud_dev   = df.groupby('device_type')['is_fraud'].mean().idxmax()
top_fraud_ctry  = df.groupby('country')['is_fraud'].mean().idxmax()
night_rate      = df[df['is_night_transaction']==1]['is_fraud'].mean() * 100
intl_rate       = df[df['is_international']==1]['is_fraud'].mean() * 100

lines = [
    "+--------------------------------------------------------------+",
    "|          BANK FRAUD DETECTION -- KEY INSIGHTS               |",
    "+--------------------------------------------------------------+",
    f"|  Total Transactions  : {len(df):,}                     |",
    f"|  Fraud Count         : {fraud_count:,} ({fraud_rate:.2f}%)              |",
    f"|  Top Fraud Type      : {top_fraud_type:<30}  |",
    f"|  Riskiest Category   : {top_fraud_mc:<30}  |",
    f"|  Riskiest Payment    : {top_fraud_pm:<30}  |",
    f"|  Riskiest Device     : {top_fraud_dev:<30}  |",
    f"|  Riskiest Country    : {top_fraud_ctry:<30}  |",
    f"|  Night Txn Fraud Rate: {night_rate:.2f}%                             |",
    f"|  Intl Txn Fraud Rate : {intl_rate:.2f}%                             |",
    "+--------------------------------------------------------------+",
]
print(chr(10).join(lines))


---
## EDA Complete!

**Next Steps:**
- Run the Streamlit dashboard: `streamlit run app.py`
- Star this repo on GitHub!

> Built with Python, Pandas, Matplotlib & Seaborn
